# U.S. Avocado Market Analysis: 2025 vs. 2026
This notebook verifies the figures used in the management memo. Because the 2026 file only contains Periods 1–32, all year-over-year comparisons use Periods 1–32 for both years.

In [1]:
import pandas as pd
import numpy as np

files = {
    2025: "HAB-FlatFile-Weekly-2025.csv",
    2026: "HAB-FlatFile-Weekly-2026-P1-P8.csv"
}

numeric_cols = [
    "ASP Current Year", "Total Bulk and Bags", "4046", "4225", "4770",
    "TotalBagged", "Sml Bagged", "Lrg Bagged", "X-Lrg Bagged", "Unknown Bag Size"
]

frames = []
for year, file in files.items():
    df = pd.read_csv(file)
    df["Year"] = year
    df["Current Year Week Ending"] = pd.to_datetime(df["Current Year Week Ending"])
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

# Same national scope and same 32 periods in both years
us = data[(data["Geography"] == "Total U.S.") & (data["Period"].between(1, 32))].copy()

us.groupby("Year")["Period"].nunique()


Year
2025    32
2026    32
Name: Period, dtype: int64

In [2]:
def weighted_asp(group):
    return np.average(group["ASP Current Year"], weights=group["Total Bulk and Bags"])

national = (
    us.groupby("Year")
      .apply(lambda g: pd.Series({
          "Total Volume": g["Total Bulk and Bags"].sum(),
          "Weighted ASP": weighted_asp(g),
          "Bagged Volume": g["TotalBagged"].sum(),
          "Bagged Share": g["TotalBagged"].sum() / g["Total Bulk and Bags"].sum(),
          "4046": g["4046"].sum(),
          "4225": g["4225"].sum(),
          "4770": g["4770"].sum(),
          "Large Bagged": g["Lrg Bagged"].sum(),
          "XL Bagged": g["X-Lrg Bagged"].sum()
      }), include_groups=False)
)

national


,Total Volume,Weighted ASP,Bagged Volume,Bagged Share,4046,4225,4770,Large Bagged,XL Bagged
Year,,,,,,,,,
2025,1.870699e+09,1.245151,7.823154e+08,0.418194,8.671604e+08,1.972775e+08,2.394555e+07,6.123515e+07,6.746146e+06
2026,2.125800e+09,1.007930,9.197537e+08,0.432662,8.890511e+08,2.675278e+08,4.946765e+07,9.632801e+07,2.126751e+07


In [3]:
by_type = (
    us.groupby(["Year", "Type"])
      .apply(lambda g: pd.Series({
          "Volume": g["Total Bulk and Bags"].sum(),
          "Weighted ASP": weighted_asp(g)
      }), include_groups=False)
      .reset_index()
)

by_type


,Year,Type,Volume,Weighted ASP
0,2025,Conventional,1.764839e+09,1.215428
1,2025,Organic,1.058595e+08,1.740676
2,2026,Conventional,2.020685e+09,0.979122
3,2026,Organic,1.051156e+08,1.561718


In [4]:
def pct_change(old, new):
    return (new / old - 1) * 100

checks = pd.DataFrame({
    "Memo figure": [
        "Total U.S. volume change",
        "Weighted ASP change",
        "Conventional volume change",
        "Organic volume change",
        "Bagged volume change",
        "Bagged share change (percentage points)",
        "PLU 4046 volume change",
        "PLU 4225 volume change",
        "PLU 4770 volume change",
        "Large bagged volume change",
        "XL bagged volume change"
    ],
    "Python result": [
        pct_change(national.loc[2025, "Total Volume"], national.loc[2026, "Total Volume"]),
        pct_change(national.loc[2025, "Weighted ASP"], national.loc[2026, "Weighted ASP"]),
        pct_change(
            by_type.query("Year == 2025 and Type == 'Conventional'")["Volume"].iloc[0],
            by_type.query("Year == 2026 and Type == 'Conventional'")["Volume"].iloc[0]
        ),
        pct_change(
            by_type.query("Year == 2025 and Type == 'Organic'")["Volume"].iloc[0],
            by_type.query("Year == 2026 and Type == 'Organic'")["Volume"].iloc[0]
        ),
        pct_change(national.loc[2025, "Bagged Volume"], national.loc[2026, "Bagged Volume"]),
        (national.loc[2026, "Bagged Share"] - national.loc[2025, "Bagged Share"]) * 100,
        pct_change(national.loc[2025, "4046"], national.loc[2026, "4046"]),
        pct_change(national.loc[2025, "4225"], national.loc[2026, "4225"]),
        pct_change(national.loc[2025, "4770"], national.loc[2026, "4770"]),
        pct_change(national.loc[2025, "Large Bagged"], national.loc[2026, "Large Bagged"]),
        pct_change(national.loc[2025, "XL Bagged"], national.loc[2026, "XL Bagged"])
    ]
})

checks["Python result"] = checks["Python result"].round(1)
checks["Verified"] = "✓"
checks


,Memo figure,Python result,Verified
0,Total U.S. volume change,13.6,✓
1,Weighted ASP change,-19.1,✓
2,Conventional volume change,14.5,✓
3,Organic volume change,-0.7,✓
4,Bagged volume change,17.6,✓
5,Bagged share change (percentage points),1.4,✓
6,PLU 4046 volume change,2.5,✓
7,PLU 4225 volume change,35.6,✓
8,PLU 4770 volume change,106.6,✓
9,Large bagged volume change,57.3,✓


In [5]:
weekly = (
    us.groupby(["Year", "Period"])
      .apply(lambda g: pd.Series({
          "Volume": g["Total Bulk and Bags"].sum(),
          "Weighted ASP": weighted_asp(g)
      }), include_groups=False)
      .reset_index()
      .pivot(index="Period", columns="Year")
)

print("Weeks 2026 volume exceeded 2025:", (weekly[("Volume", 2026)] > weekly[("Volume", 2025)]).sum(), "of 32")
print("Weeks 2026 weighted ASP was below 2025:", (weekly[("Weighted ASP", 2026)] < weekly[("Weighted ASP", 2025)]).sum(), "of 32")


Weeks 2026 volume exceeded 2025: 32 of 32
Weeks 2026 weighted ASP was below 2025: 32 of 32
